# CWRU fault classifier with a load-held-out test set

This notebook compares four classes from drive-end vibration: normal, inner-race, ball, and centered outer-race faults. We use 0–2 HP recordings for training and reserve every 3 HP recording for the final test. All segments from a recording inherit that recording's split.

The fault recordings cover 0.007, 0.014, and 0.021 inch defects. The reusable signal features live in `ml/src/motorsense_ml`; this notebook handles CWRU file loading, plots, and model evaluation.


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from scipy.io import loadmat
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import ConfusionMatrixDisplay, balanced_accuracy_score, confusion_matrix
from sklearn.model_selection import StratifiedGroupKFold, cross_val_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

from motorsense_ml.dataset import segment_signal
from motorsense_ml.features import crest_factor, envelope_band_energies, kurtosis, rms
from motorsense_ml.resampling import resample_signal

ROOT = next(path for path in (Path.cwd(), *Path.cwd().parents) if (path / "ml" / "data").is_dir())
DATA_DIR = ROOT / "ml" / "data"
TARGET_FS = 12_000
SOURCE_FS_BY_CLASS = {"normal": 48_000, "inner race": 12_000, "ball": 12_000, "outer race": 12_000}
RPM_BY_LOAD = {0: 1797, 1: 1772, 2: 1750, 3: 1730}  # CWRU table approximations
SEGMENT_SAMPLES = TARGET_FS // 2  # half-second windows; 2 Hz envelope FFT bins
SEGMENTS_PER_RECORDING = 10
FAULT_IDS = {
    "inner race": {
        0.007: (105, 106, 107, 108),
        0.014: (169, 170, 171, 172),
        0.021: (209, 210, 211, 212),
    },
    "ball": {0.007: (118, 119, 120, 121), 0.014: (185, 186, 187, 188), 0.021: (222, 223, 224, 225)},
    "outer race": {
        0.007: (130, 131, 132, 133),
        0.014: (197, 198, 199, 200),
        0.021: (234, 235, 236, 237),
    },
}

## Check the normal recordings' sample-rate assumption

The CWRU pages document the normal files by load and say the drive-end collection includes 12 kHz and 48 kHz data, but they do not give a per-file normal-baseline rate. Decision 001 verified `97.mat` at 48 kHz by comparing its shaft harmonics with the known 12 kHz `105.mat`. We treat `98.mat`–`100.mat` as the same 48 kHz normal-baseline collection; their sample counts are about 484k, consistent with a roughly 10 s 48 kHz record. This is a collection-level assumption, not explicit per-file metadata. The code reports duration under each candidate so the assumption remains visible.


In [ ]:
def drive_end_signal(file_id):
    mat = loadmat(DATA_DIR / f"{file_id}.mat")
    key = f"X{file_id:03d}_DE_time"
    if key not in mat:
        raise KeyError(f"{key} missing from {file_id}.mat")
    return np.asarray(mat[key], dtype=float).reshape(-1)


for load_hp, file_id in enumerate((97, 98, 99, 100)):
    signal = drive_end_signal(file_id)
    print(
        f"{file_id}.mat ({load_hp} HP): {signal.size:,} samples; "
        f"{signal.size / 48_000:.2f} s at 48 kHz or {signal.size / 12_000:.2f} s at 12 kHz"
    )

The sample-count check above does not establish the rate by itself: duration depends on the assumed rate. We use 48 kHz for the normal-baseline collection because the CWRU record set is the 48 kHz normal baseline; the official page does not provide per-file rate metadata. The existing `97.mat` harmonic comparison is documented in Decision 001. We resample the normal signals to 12 kHz so their feature windows match the 12 kHz fault recordings.


## Build fixed-length feature rows

Each row represents one non-overlapping 0.5 s segment. We keep ten evenly spaced segments from each recording so one long file does not dominate. The split is assigned from `load_hp` after features are extracted, and the test set is the entire 3 HP recording for each class/size. No recording contributes segments to both sides.

Envelope energies sum squared envelope-spectrum magnitudes around the first three harmonics of BPFI, BPFO, BSF, and FTF. Those orders are based on the CWRU drive-end 6205 bearing. The feature helper returns them in that fixed order on a log scale.


In [ ]:
recording_specs = []
for file_id, load_hp in zip((97, 98, 99, 100), range(4), strict=True):
    recording_specs.append(
        {
            "file_id": file_id,
            "label": "normal",
            "load_hp": load_hp,
            "fault_size_in": 0.0,
            "outer_position": None,
        }
    )
for label, sizes in FAULT_IDS.items():
    for size, file_ids in sizes.items():
        for load_hp, file_id in enumerate(file_ids):
            recording_specs.append(
                {
                    "file_id": file_id,
                    "label": label,
                    "load_hp": load_hp,
                    "fault_size_in": size,
                    "outer_position": "centered @ 6" if label == "outer race" else None,
                }
            )

feature_names = [
    "rms",
    "kurtosis",
    "crest_factor",
    "log_bpfi_energy",
    "log_bpfo_energy",
    "log_bsf_energy",
    "log_ftf_energy",
]
feature_rows = []
recording_rows = []
for spec in recording_specs:
    file_id = spec["file_id"]
    signal = drive_end_signal(file_id)
    mat = loadmat(DATA_DIR / f"{file_id}.mat")
    rpm_key = f"X{file_id:03d}RPM"
    rpm = (
        float(np.asarray(mat[rpm_key]).squeeze())
        if rpm_key in mat
        else RPM_BY_LOAD[spec["load_hp"]]
    )
    source_fs = SOURCE_FS_BY_CLASS[spec["label"]]
    if source_fs != TARGET_FS:
        signal = resample_signal(signal, source_fs, TARGET_FS)
    segments = segment_signal(signal, SEGMENT_SAMPLES, max_segments=SEGMENTS_PER_RECORDING)
    recording_id = f"{file_id:03d}"
    recording_rows.append(
        {
            **spec,
            "recording_id": recording_id,
            "rpm_used": rpm,
            "source_fs": source_fs,
            "segment_count": len(segments),
        }
    )
    for segment_index, segment in enumerate(segments):
        env = envelope_band_energies(segment, TARGET_FS, rpm / 60)
        values = [rms(segment), kurtosis(segment), crest_factor(segment), *env]
        feature_rows.append(
            {
                "recording_id": recording_id,
                "segment_index": segment_index,
                "label": spec["label"],
                "load_hp": spec["load_hp"],
                "fault_size_in": spec["fault_size_in"],
                **dict(zip(feature_names, values, strict=True)),
            }
        )

print(f"{len(recording_rows)} recordings, {len(feature_rows)} segments")
for load_hp in range(4):
    counts = {
        label: sum(row["load_hp"] == load_hp and row["label"] == label for row in recording_rows)
        for label in ("normal", "inner race", "ball", "outer race")
    }
    print(f"{load_hp} HP: {counts}")
print("Example feature row:", feature_rows[0])

## Split by motor load, then check recording IDs

Loads 0, 1, and 2 HP form the training pool. Load 3 HP is held out in full. Grouped cross-validation inside the training pool also keeps each recording together while choosing between the two simple classifiers.


In [ ]:
train = [row for row in feature_rows if row["load_hp"] < 3]
test = [row for row in feature_rows if row["load_hp"] == 3]
train_recordings = {row["recording_id"] for row in train}
test_recordings = {row["recording_id"] for row in test}
assert train_recordings.isdisjoint(test_recordings)
assert {row["load_hp"] for row in train} == {0, 1, 2}
assert {row["load_hp"] for row in test} == {3}
print(f"Train: {len(train_recordings)} recordings / {len(train)} segments")
print(f"Test:  {len(test_recordings)} recordings / {len(test)} segments")
print("Overlapping recording IDs:", train_recordings & test_recordings)

X_train = np.asarray([[row[name] for name in feature_names] for row in train])
y_train = np.asarray([row["label"] for row in train])
X_test = np.asarray([[row[name] for name in feature_names] for row in test])
labels = ["normal", "inner race", "ball", "outer race"]
groups = np.asarray([row["recording_id"] for row in train])
cv = StratifiedGroupKFold(n_splits=3, shuffle=True, random_state=42)
models = {
    "dummy prior": DummyClassifier(strategy="prior"),
    "logistic regression": make_pipeline(
        StandardScaler(),
        LogisticRegression(class_weight="balanced", max_iter=2_000, random_state=42),
    ),
    "random forest": RandomForestClassifier(
        n_estimators=300, min_samples_leaf=2, class_weight="balanced", random_state=42, n_jobs=-1
    ),
}
cv_rows = []
for name, estimator in models.items():
    scores = cross_val_score(
        estimator, X_train, y_train, groups=groups, cv=cv, scoring="balanced_accuracy"
    )
    cv_rows.append(
        {"model": name, "fold_scores": scores.round(3), "mean": scores.mean(), "std": scores.std()}
    )
for row in sorted(cv_rows, key=lambda result: result["mean"], reverse=True):
    print(
        f"{row['model']}: folds={row['fold_scores']}, mean={row['mean']:.3f}, std={row['std']:.3f}"
    )

Cross-validation uses only the 0–2 HP training pool, grouped by recording. We select the non-dummy model with the higher mean balanced accuracy, then fit it on all training recordings. The 3 HP recordings remain untouched until the final confusion matrix.


In [ ]:
best_result = max(
    (row for row in cv_rows if row["model"] != "dummy prior"), key=lambda row: row["mean"]
)
best_name = best_result["model"]
model = models[best_name].fit(X_train, y_train)
print("Selected from training-only grouped CV:", best_name)

# Average segment scores, then count one prediction per recording.
segment_probabilities = model.predict_proba(X_test)
class_order = list(model.classes_)
test_by_recording = {}
for index, row in enumerate(test):
    test_by_recording.setdefault(row["recording_id"], []).append(index)
recording_predictions = []
for recording_id, row_indices in test_by_recording.items():
    mean_probability = segment_probabilities[row_indices].mean(axis=0)
    row = test[row_indices[0]]
    recording_predictions.append(
        {
            "recording_id": recording_id,
            "actual": row["label"],
            "predicted": class_order[int(np.argmax(mean_probability))],
            "load_hp": row["load_hp"],
            "fault_size_in": row["fault_size_in"],
        }
    )
actual = [row["actual"] for row in recording_predictions]
predicted = [row["predicted"] for row in recording_predictions]
score = balanced_accuracy_score(actual, predicted)
print(f"3 HP recording-level balanced accuracy: {score:.3f}")
for row in recording_predictions:
    print(row)

In [ ]:
matrix = confusion_matrix(actual, predicted, labels=labels)
display = ConfusionMatrixDisplay(matrix, display_labels=labels)
display.plot(cmap="Blues", values_format="d", colorbar=False)
plt.title("CWRU classifier: held-out 3 HP recordings")
plt.xticks(rotation=25, ha="right")
plt.tight_layout()
plt.show()

## Interpretation limits

This is a small, controlled benchmark: there is one 3 HP normal recording and one recording per fault-size/load combination. The split tests transfer across load while retaining the same fault sizes in training. It does not establish performance on a new motor, a different sensor mount, unobserved defect sizes, or real early-stage faults. The repeated segments are useful for fitting, but the final matrix counts independent recordings.
